In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel


In [ ]:
#Lectura del archivo de ENSAMBLE para obtener los valores de f1 score para el ensamble final
ensamble = pd.read_csv("outputs/ENSEMBLE/kfold_output.csv")

#Se usa solo el modelo de Stacking Ensemble
ensamble_filter = ensamble[ensamble['model']=='Stacking Ensemble']

#Se guarda el archivo para analisis
ensamble_filter.to_csv("outputs/ENSEMBLE/kfold_results.csv", index=False)

In [ ]:
#Lectura del archivo TABPFN para obtener los valores de f1 score
tabpfn = pd.read_csv("outputs/TABPFN/fold_metrics.csv")

#Se agrupa la información para obtener el dataframe final con f1 score promedio
tabpfn_grouped = (
    tabpfn
    .groupby("fold")[["accuracy", "precision", "recall", "f1", "roc_auc"]]
    .mean()
    .reset_index()
)

#Guardamos el dataframe para analisis
tabpfn_grouped.to_csv("outputs/TABPFN/tabpfn_results.csv", index=False)

In [ ]:
#Lectura del archivo TABNET para obtener los valores de f1 score
tabnet = pd.read_csv("outputs/TABNET/fold_result.csv")

#Se agrupa la información para obtener el dataframe final con f1 score promedio
tabnet_grouped = (
    tabnet
    .groupby("fold")[["accuracy", "precision", "recall", "f1", "roc_auc"]]
    .mean()
    .reset_index()
)

#Guardamos el dataframe para analisis
tabnet_grouped.to_csv("outputs/TABNET/tabnet_results.csv", index=False)

In [ ]:
model_folders = {
    "XG BOOST": "outputs/XG BOOST",
    "ENSAMBLE": "outputs/ENSAMBLE",
    "TABPFN": "outputs/TABPFN",
    "TABNET": "outputs/TABNET",
}

def load_f1_scores(folder_path, metric_col="f1"):
    all_files = glob.glob(os.path.join(folder_path, "*_results.csv"))
    if not all_files:
        raise FileNotFoundError(f"No se encontraron archivos '_results.csv' en {folder_path}")

    dfs = [pd.read_csv(f) for f in all_files]
    df = pd.concat(dfs, ignore_index=True)

    if metric_col not in df.columns:
        raise ValueError(f"La columna '{metric_col}' no existe en {folder_path}. Columnas disponibles: {df.columns.tolist()}")

    return df[metric_col].values

# Cargar resultados
results = {model: load_f1_scores(path) for model, path in model_folders.items()}

# Función para prueba de hipótesis unilateral
def one_sided_ttest(base, compare, alpha=0.05):
    min_len = min(len(base), len(compare))  # asegurar mismo tamaño
    t_stat, p_val_two_sided = ttest_rel(base[:min_len], compare[:min_len])
    p_val_one_sided = p_val_two_sided / 2 if t_stat > 0 else 1.0
    return t_stat, p_val_one_sided, p_val_one_sided < alpha

# F1 de Ensemble
ensemble_scores = results["ENSAMBLE"]
ensemble_mean = np.mean(ensemble_scores)
ensemble_std = np.std(ensemble_scores)
ensemble_summary = f"{ensemble_mean:.4f} ({ensemble_std:.4f})"

print("### Resultados Pruebas de Hipótesis ###\n")
summary = []

# Agregar TabNet como referencia al inicio del resumen
summary.append(["ENSEMBLE (baseline)", ensemble_summary, np.nan, np.nan, "—"])

# Comparar TabNet contra los demás
for model_name, scores in results.items():
    if model_name == "ENSAMBLE":
        continue

    # Mostrar hipótesis antes de cada test
    print(f"\nComparación: Ensemble vs {model_name}")
    print(f"H0: F1(TabNet) ≤ F1({model_name})")
    print(f"H1: F1(TabNet) > F1({model_name})")

    # Ejecutar test
    t, p, reject = one_sided_ttest(ensemble_scores, scores)
    mean_f1 = np.mean(scores)
    std_f1 = np.std(scores)
    formatted = f"{mean_f1:.4f} ({std_f1:.4f})"

    summary.append([model_name, formatted, t, p, reject])
    print(f"t={t:.3f}, p={p:.4f}, Rechazar H0? {reject} | F1 promedio = {formatted}")

# Resumen en DataFrame
summary_df = pd.DataFrame(summary, columns=["Modelo", "F1 (media ± std)", "t", "p-value", "Rechaza H0?"])
print("\n--- Resumen Comparaciones ---")
print(summary_df)

# Guardar resultados
os.makedirs("outputs/HIPOTESIS", exist_ok=True)
summary_df.to_csv("outputs/HIPOTESIS/resultados_pruebas.csv", index=False)
print("\nArchivo guardado en outputs/HIPOTESIS/resultados_pruebas.csv")


### Resultados Pruebas de Hipótesis ###


Comparación: Ensemble vs XG BOOST
H0: F1(TabNet) ≤ F1(XG BOOST)
H1: F1(TabNet) > F1(XG BOOST)
t=155.543, p=0.0000, Rechazar H0? True | F1 promedio = 0.4398 (0.0034)

Comparación: Ensemble vs TABPFN
H0: F1(TabNet) ≤ F1(TABPFN)
H1: F1(TabNet) > F1(TABPFN)
t=195.200, p=0.0000, Rechazar H0? True | F1 promedio = 0.3259 (0.0025)

Comparación: Ensemble vs TABNET
H0: F1(TabNet) ≤ F1(TABNET)
H1: F1(TabNet) > F1(TABNET)
t=368.961, p=0.0000, Rechazar H0? True | F1 promedio = 0.4350 (0.0048)

--- Resumen Comparaciones ---
                Modelo F1 (media ± std)           t       p-value Rechaza H0?
0  ENSEMBLE (baseline)  0.8018 (0.0035)         NaN           NaN           —
1             XG BOOST  0.4398 (0.0034)  155.542575  5.123945e-09        True
2               TABPFN  0.3259 (0.0025)  195.200265  2.065968e-09        True
3               TABNET  0.4350 (0.0048)  368.960759  1.618748e-10        True

Archivo guardado en HIPOTESIS/outputs/resultados_pr

In [ ]:
model_folders = {
    "XG BOOST": "outputs/XG BOOST",
    "ENSAMBLE": "outputs/ENSAMBLE",
    "TABPFN": "outputs/TABPFN",
    "TABNET": "outputs/TABNET",
}

def load_all_metrics(folder_path):
    all_files = glob.glob(os.path.join(folder_path, "*_results.csv"))
    if not all_files:
        raise FileNotFoundError(f"No se encontraron archivos '_results.csv' en {folder_path}")
    dfs = [pd.read_csv(f) for f in all_files]
    return pd.concat(dfs, ignore_index=True)

# Crear tabla resumen
summary_metrics = {}

for model, path in model_folders.items():
    df = load_all_metrics(path)
    # Calcular media y desviación estándar solo para columnas numéricas
    means = df.mean(numeric_only=True)
    stds = df.std(numeric_only=True)

    # Combinar en formato "media (std)"
    metrics_summary = {col: f"{means[col]:.4f} ({stds[col]:.4f})" for col in means.index}
    summary_metrics[model] = metrics_summary

# Convertir a DataFrame
summary_df = pd.DataFrame(summary_metrics).T.reset_index()
summary_df.rename(columns={"index": "Modelo"}, inplace=True)

print("\n--- Tabla resumen de métricas por modelo ---")
print(summary_df)

# Guardar resultados
os.makedirs("outputs/HIPOTESIS", exist_ok=True)
summary_df.to_csv("outputs/HIPOTESIS/resumen_metricas.csv", index=False)
print("\nArchivo guardado en outputs/HIPOTESIS/resumen_metricas.csv")



--- Tabla resumen de métricas por modelo ---
     Modelo             fold         accuracy        precision  \
0  XG BOOST  3.0000 (1.5811)  0.5891 (0.0039)  0.3034 (0.0029)   
1  ENSAMBLE  3.0000 (1.5811)  0.8029 (0.0034)  0.8036 (0.0034)   
2    TABPFN  2.0000 (1.5811)  0.7856 (0.0021)  0.4312 (0.0072)   
3    TABNET  3.0000 (1.5811)  0.5485 (0.0233)  0.2917 (0.0075)   

            recall               f1          roc_auc  
0  0.7987 (0.0059)  0.4398 (0.0038)  0.7239 (0.0047)  
1  0.7999 (0.0069)  0.8018 (0.0040)  0.8866 (0.0023)  
2  0.2708 (0.0040)  0.3259 (0.0028)  0.7347 (0.0043)  
3  0.8591 (0.0264)  0.4350 (0.0054)  0.7181 (0.0026)  

Archivo guardado en HIPOTESIS/outputs/resumen_metricas.csv
